In [ ]:
import json
import os
import pickle
import time
from pathlib import Path

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import optax


def environment_flag(name, default):
    return os.environ.get(name, str(default)).strip().lower() in {
        "1", "true", "yes", "on"
    }


ENABLE_X64 = environment_flag("JAX_E65_X64", True)
jax.config.update("jax_enable_x64", ENABLE_X64)

# ==============================================================================
# 1. E65 data and experiment configuration
# ==============================================================================

SEED = int(os.environ.get("JAX_E65_SEED", "42"))
DATA_PATH = Path(
    os.environ.get("JAX_E65_DATA_PATH", "src/npz_e65_data/E65_data.npz")
).expanduser()
OUTPUT_DIR = Path(
    os.environ.get("JAX_E65_OUT_DIR", "runs/jax_tmp_e65_geodesic_vs_free")
).expanduser()
HELDOUT_FRACTION = float(os.environ.get("JAX_E65_HELDOUT_FRAC", "0.20"))
MAX_TRIALS = int(os.environ.get("JAX_E65_MAX_TRIALS", "0"))
CONTEXT_STEPS = int(os.environ.get("JAX_E65_CONTEXT_STEPS", "10"))
EPOCHS = int(os.environ.get("JAX_E65_EPOCHS", "2000"))
BATCH_SIZE = int(os.environ.get("JAX_E65_BATCH_SIZE", "16"))
LR = float(os.environ.get("JAX_E65_LR", "0.001"))
LOG_EVERY = int(os.environ.get("JAX_E65_LOG_EVERY", "100"))
LATENT_DIM = int(os.environ.get("JAX_E65_LATENT_DIM", "5"))
ENCODER_TYPE = os.environ.get("JAX_E65_ENCODER", "flattened_mlp").strip().lower()
GRU_HIDDEN_DIM = int(os.environ.get("JAX_E65_GRU_HIDDEN_DIM", "128"))
CHECKPOINT_EVERY = int(os.environ.get("JAX_E65_CHECKPOINT_EVERY", "50"))
RESUME = environment_flag("JAX_E65_RESUME", True)
REQUIRE_GPU = environment_flag("JAX_E65_REQUIRE_GPU", False)
if ENCODER_TYPE not in {"flattened_mlp", "gru"}:
    raise ValueError("JAX_E65_ENCODER must be 'flattened_mlp' or 'gru'.")
if GRU_HIDDEN_DIM <= 0:
    raise ValueError("JAX_E65_GRU_HIDDEN_DIM must be positive.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def split_trials(trial_ids, heldout_fraction, seed):
    """Reproducible 80/20 split of complete trials, never individual frames."""
    rng = np.random.default_rng(seed)
    shuffled = rng.permutation(len(trial_ids))
    if heldout_fraction <= 0.0 or len(trial_ids) == 1:
        return shuffled, np.empty(0, dtype=np.int64)
    n_heldout = max(1, int(round(heldout_fraction * len(trial_ids))))
    n_heldout = min(n_heldout, len(trial_ids) - 1)
    return shuffled[n_heldout:], shuffled[:n_heldout]


def load_e65(npz_path, heldout_fraction, seed, max_trials=0):
    """Load raw E65 activity and pad variable-length trials for JAX batching."""
    with np.load(npz_path) as source:
        raw_activity = np.asarray(source["roi"], dtype=np.float64).T
        frame_trial = np.asarray(source["Trial"]).reshape(-1)
        frame_time = np.asarray(source["Time"], dtype=np.float64).reshape(-1)

    trial_ids = np.unique(frame_trial)
    if max_trials < 0:
        raise ValueError("JAX_E65_MAX_TRIALS must be nonnegative.")
    if max_trials and max_trials < len(trial_ids):
        subset_rng = np.random.default_rng(seed)
        trial_ids = np.sort(
            subset_rng.choice(trial_ids, size=max_trials, replace=False)
        )
    train_indices, heldout_indices = split_trials(
        trial_ids, heldout_fraction, seed
    )

    # Decide which neurons exist using training data only. This avoids test leakage
    # and removes neurons for which the model receives no training information.
    train_trial_ids = trial_ids[train_indices]
    train_frame_mask = np.isin(frame_trial, train_trial_ids)
    neuron_mask = np.any(raw_activity[train_frame_mask] > 0.0, axis=0)
    if not np.any(neuron_mask):
        raise ValueError("Train-silent filtering removed every neuron.")

    trial_activity = []
    trial_times = []
    for trial_id in trial_ids:
        frame_mask = frame_trial == trial_id
        activity = raw_activity[frame_mask][:, neuron_mask]
        times = frame_time[frame_mask]
        order = np.argsort(times, kind="stable")
        activity = activity[order]
        times = times[order] - times[order][0]

        differences = np.diff(times)
        if np.any(differences <= 0):
            positive = differences[differences > 0]
            replacement = float(np.median(positive)) if len(positive) else 1.0
            differences = np.where(differences > 0, differences, replacement)
            times = np.concatenate([[0.0], np.cumsum(differences)])

        trial_activity.append(activity)
        trial_times.append(times)

    lengths = np.asarray([len(values) for values in trial_activity], dtype=np.int64)
    max_length = int(lengths.max())
    n_trials = len(trial_ids)
    n_neurons = int(neuron_mask.sum())

    padded_activity = np.zeros(
        (n_trials, max_length, n_neurons), dtype=np.float64
    )
    padded_times = np.zeros((n_trials, max_length), dtype=np.float64)
    valid_mask = np.zeros((n_trials, max_length), dtype=bool)
    time_steps = np.zeros((n_trials, max_length - 1), dtype=np.float64)

    for index, (activity, times) in enumerate(zip(trial_activity, trial_times)):
        length = len(activity)
        padded_activity[index, :length] = activity
        padded_times[index, :length] = times
        valid_mask[index, :length] = True
        time_steps[index, : length - 1] = np.diff(times)

    return {
        "activity": padded_activity,
        "times": padded_times,
        "time_steps": time_steps,
        "valid_mask": valid_mask,
        "lengths": lengths,
        "trial_ids": trial_ids,
        "train_indices": train_indices.astype(np.int64),
        "heldout_indices": heldout_indices.astype(np.int64),
        "neuron_mask": neuron_mask,
        "n_neurons_raw": raw_activity.shape[1],
    }


e65 = load_e65(DATA_PATH, HELDOUT_FRACTION, SEED, MAX_TRIALS)
if not 2 <= CONTEXT_STEPS < int(e65["lengths"][e65["heldout_indices"]].min()):
    raise ValueError(
        "CONTEXT_STEPS must be at least 2 and shorter than every heldout trial."
    )

devices = jax.devices()
print(f"JAX devices: {devices}")
if REQUIRE_GPU and not any(device.platform == "gpu" for device in devices):
    raise RuntimeError(
        "JAX_E65_REQUIRE_GPU is enabled, but JAX did not find a GPU."
    )
print(
    f"E65: {len(e65['trial_ids'])} trials, "
    f"train={len(e65['train_indices'])}, "
    f"heldout={len(e65['heldout_indices'])}"
)
print(
    f"Trial lengths: {e65['lengths'].min()}-{e65['lengths'].max()} frames; "
    f"neurons: {e65['n_neurons_raw']} raw -> {e65['neuron_mask'].sum()} modeled"
)
print(
    f"Encoder input: first {CONTEXT_STEPS} neural frames; "
    f"heldout forecast begins at frame {CONTEXT_STEPS}."
)
print(
    f"Latent dimensions: {LATENT_DIM}; precision: "
    f"{'float64' if ENABLE_X64 else 'float32'}; resume: {RESUME}."
)
print(f"Initial-condition encoder: {ENCODER_TYPE}.")

In [ ]:
# ==============================================================================
# 2. Original learned-geodesic model plus a matched free-dynamics comparison
# ==============================================================================

def init_mlp(keys, layer_sizes):
    """Original MLP initialization."""
    params = []
    for i in range(len(layer_sizes) - 1):
        weights = (
            jax.random.normal(keys[i], (layer_sizes[i], layer_sizes[i + 1]))
            * jnp.sqrt(2.0 / layer_sizes[i])
        )
        bias = jnp.zeros((layer_sizes[i + 1],))
        params.append((weights, bias))
    return params


def apply_mlp(params, values, final_activation=None):
    """Original swish MLP forward pass."""
    for weights, bias in params[:-1]:
        values = jax.nn.swish(jnp.dot(values, weights) + bias)
    weights, bias = params[-1]
    output = jnp.dot(values, weights) + bias
    if final_activation == "softplus":
        return jax.nn.softplus(output)
    return output


def cholesky_size(latent_dim):
    return latent_dim * (latent_dim + 1) // 2


def init_gru_encoder(key, n_neurons):
    """Initialize a GRU that maps an ordered neural prefix to z0 and v0."""
    input_key, hidden_key, readout_key = jax.random.split(key, 3)
    gate_width = 3 * GRU_HIDDEN_DIM
    return {
        "input_weights": (
            jax.random.normal(input_key, (n_neurons, gate_width))
            * jnp.sqrt(1.0 / n_neurons)
        ),
        "hidden_weights": (
            jax.random.normal(hidden_key, (GRU_HIDDEN_DIM, gate_width))
            * jnp.sqrt(1.0 / GRU_HIDDEN_DIM)
        ),
        "bias": jnp.zeros((gate_width,)),
        "readout": init_mlp(
            jax.random.split(readout_key, 2),
            [GRU_HIDDEN_DIM, 128, 2 * LATENT_DIM],
        ),
    }


def apply_gru_encoder(params, neural_context):
    """Read context frames in order and return inferred initial conditions."""
    def step(hidden, neural_frame):
        input_gates = neural_frame @ params["input_weights"] + params["bias"]
        hidden_gates = hidden @ params["hidden_weights"]
        input_reset, input_update, input_candidate = jnp.split(input_gates, 3)
        hidden_reset, hidden_update, hidden_candidate = jnp.split(hidden_gates, 3)
        reset = jax.nn.sigmoid(input_reset + hidden_reset)
        update = jax.nn.sigmoid(input_update + hidden_update)
        candidate = jnp.tanh(input_candidate + reset * hidden_candidate)
        next_hidden = update * hidden + (1.0 - update) * candidate
        return next_hidden, None

    initial_hidden = jnp.zeros((GRU_HIDDEN_DIM,))
    final_hidden, _ = jax.lax.scan(step, initial_hidden, neural_context)
    return apply_mlp(params["readout"], final_hidden)


def init_geodesic_params(key, n_neurons, context_steps):
    """Original encoder/metric/decoder structure generalized to LATENT_DIM."""
    keys = jax.random.split(key, 3)

    if ENCODER_TYPE == "gru":
        encoder = init_gru_encoder(keys[0], n_neurons)
    else:
        encoder_keys = jax.random.split(keys[0], 4)
        encoder = init_mlp(
            encoder_keys,
            [n_neurons * context_steps, 128, 128, 2 * LATENT_DIM],
        )

    decoder_keys = jax.random.split(keys[1], 4)
    decoder = init_mlp(
        decoder_keys, [LATENT_DIM, 128, 128, n_neurons]
    )

    metric_keys = jax.random.split(keys[2], 4)
    n_metric_elements = cholesky_size(LATENT_DIM)
    metric = init_mlp(
        metric_keys, [LATENT_DIM, 64, 64, n_metric_elements]
    )

    # Initialize every Cholesky diagonal near one and every off-diagonal
    # at zero. This is the dimension-general version of the original bias.
    metric_bias = jnp.zeros((n_metric_elements,))
    diagonal_bias = jnp.log(jnp.expm1(0.9))
    element_index = 0
    for row in range(LATENT_DIM):
        for column in range(row + 1):
            if row == column:
                metric_bias = metric_bias.at[element_index].set(diagonal_bias)
            element_index += 1
    metric[-1] = (
        jnp.zeros((64, n_metric_elements)),
        metric_bias,
    )
    return {"enc": encoder, "metric": metric, "dec": decoder}


def init_free_params(key, n_neurons, context_steps):
    """Use the same initialized encoder/decoder but replace the metric by F(x,v)."""
    geodesic_template = init_geodesic_params(key, n_neurons, context_steps)
    dynamics_key = jax.random.split(jax.random.split(key, 3)[2], 4)
    dynamics = init_mlp(
        dynamics_key, [2 * LATENT_DIM, 64, 64, LATENT_DIM]
    )
    return {
        "enc": geodesic_template["enc"],
        "dynamics": dynamics,
        "dec": geodesic_template["dec"],
    }


def get_metric(metric_params, latent_position):
    """Learn a positive-definite LATENT_DIM x LATENT_DIM metric."""
    lower_elements = apply_mlp(metric_params, latent_position)
    lower = jnp.zeros((LATENT_DIM, LATENT_DIM))
    element_index = 0
    for row in range(LATENT_DIM):
        for column in range(row + 1):
            value = lower_elements[element_index]
            if row == column:
                value = jax.nn.softplus(value) + 0.1
            lower = lower.at[row, column].set(value)
            element_index += 1
    return lower @ lower.T + 1e-3 * jnp.eye(LATENT_DIM)


def get_christoffel(metric_params, latent_position):
    """Original explicit Christoffel-symbol calculation."""
    metric = get_metric(metric_params, latent_position)
    inverse_metric = jnp.linalg.inv(metric)

    # dg[i,j,k] = partial_k g_ij
    metric_jacobian = jax.jacobian(get_metric, argnums=1)(
        metric_params, latent_position
    )
    derivatives = jnp.transpose(metric_jacobian, (2, 0, 1))

    term_1 = jnp.einsum("imj->mij", derivatives)
    term_2 = jnp.einsum("jmi->mij", derivatives)
    term_3 = jnp.einsum("mij->mij", derivatives)
    christoffel_first_kind = 0.5 * (term_1 + term_2 - term_3)
    return jnp.einsum(
        "mk,kij->mij", inverse_metric, christoffel_first_kind
    )


def geodesic_acceleration(latent_position, latent_velocity, metric_params):
    christoffel = get_christoffel(metric_params, latent_position)
    return -jnp.einsum(
        "mij,i,j->m", christoffel, latent_velocity, latent_velocity
    )


def free_acceleration(latent_position, latent_velocity, dynamics_params):
    state = jnp.concatenate([latent_position, latent_velocity])
    return apply_mlp(dynamics_params, state)


def rk4_geodesic_step(position, velocity, dt, metric_params):
    """Original RK4 integrator, now supplied each real E65 frame interval."""
    def derivative(state):
        current_position, current_velocity = state
        return (
            current_velocity,
            geodesic_acceleration(
                current_position, current_velocity, metric_params
            ),
        )

    k1_position, k1_velocity = derivative((position, velocity))
    k2_position, k2_velocity = derivative(
        (
            position + 0.5 * dt * k1_position,
            velocity + 0.5 * dt * k1_velocity,
        )
    )
    k3_position, k3_velocity = derivative(
        (
            position + 0.5 * dt * k2_position,
            velocity + 0.5 * dt * k2_velocity,
        )
    )
    k4_position, k4_velocity = derivative(
        (
            position + dt * k3_position,
            velocity + dt * k3_velocity,
        )
    )
    next_position = position + (dt / 6.0) * (
        k1_position + 2 * k2_position + 2 * k3_position + k4_position
    )
    next_velocity = velocity + (dt / 6.0) * (
        k1_velocity + 2 * k2_velocity + 2 * k3_velocity + k4_velocity
    )
    return next_position, next_velocity


def rk4_free_step(position, velocity, dt, dynamics_params):
    """The same RK4 solver with unconstrained learned acceleration F(x,v)."""
    def derivative(state):
        current_position, current_velocity = state
        return (
            current_velocity,
            free_acceleration(
                current_position, current_velocity, dynamics_params
            ),
        )

    k1_position, k1_velocity = derivative((position, velocity))
    k2_position, k2_velocity = derivative(
        (
            position + 0.5 * dt * k1_position,
            velocity + 0.5 * dt * k1_velocity,
        )
    )
    k3_position, k3_velocity = derivative(
        (
            position + 0.5 * dt * k2_position,
            velocity + 0.5 * dt * k2_velocity,
        )
    )
    k4_position, k4_velocity = derivative(
        (
            position + dt * k3_position,
            velocity + dt * k3_velocity,
        )
    )
    next_position = position + (dt / 6.0) * (
        k1_position + 2 * k2_position + 2 * k3_position + k4_position
    )
    next_velocity = velocity + (dt / 6.0) * (
        k1_velocity + 2 * k2_velocity + 2 * k3_velocity + k4_velocity
    )
    return next_position, next_velocity


def integrate_geodesic(initial_position, initial_velocity, time_steps, metric_params):
    def step(state, dt):
        position, velocity = state
        next_position, next_velocity = rk4_geodesic_step(
            position, velocity, dt, metric_params
        )
        return (next_position, next_velocity), next_position

    _, remaining_positions = jax.lax.scan(
        step,
        (initial_position, initial_velocity),
        time_steps,
    )
    return jnp.concatenate(
        [initial_position[None, :], remaining_positions], axis=0
    )


def integrate_free(initial_position, initial_velocity, time_steps, dynamics_params):
    def step(state, dt):
        position, velocity = state
        next_position, next_velocity = rk4_free_step(
            position, velocity, dt, dynamics_params
        )
        return (next_position, next_velocity), next_position

    _, remaining_positions = jax.lax.scan(
        step,
        (initial_position, initial_velocity),
        time_steps,
    )
    return jnp.concatenate(
        [initial_position[None, :], remaining_positions], axis=0
    )


def infer_initial_conditions(params, neural_sequence, context_steps):
    """Infer z0 and v0 from only the trial prefix."""
    neural_context = neural_sequence[:context_steps]
    if ENCODER_TYPE == "gru":
        encoded = apply_gru_encoder(params["enc"], neural_context)
    else:
        encoded = apply_mlp(params["enc"], neural_context.reshape(-1))
    return encoded[:LATENT_DIM], encoded[LATENT_DIM:]


def geodesic_model_forward(params, neural_sequence, time_steps, context_steps):
    initial_position, initial_velocity = infer_initial_conditions(
        params, neural_sequence, context_steps
    )
    latent_trajectory = integrate_geodesic(
        initial_position,
        initial_velocity,
        time_steps,
        params["metric"],
    )
    # E65 activity can exceed 1, so sigmoid would make exact reconstruction
    # impossible. Softplus is the minimal nonnegative, unbounded replacement.
    return jax.vmap(apply_mlp, in_axes=(None, 0, None))(
        params["dec"], latent_trajectory, "softplus"
    )


def free_model_forward(params, neural_sequence, time_steps, context_steps):
    initial_position, initial_velocity = infer_initial_conditions(
        params, neural_sequence, context_steps
    )
    latent_trajectory = integrate_free(
        initial_position,
        initial_velocity,
        time_steps,
        params["dynamics"],
    )
    return jax.vmap(apply_mlp, in_axes=(None, 0, None))(
        params["dec"], latent_trajectory, "softplus"
    )


def masked_mse(true_activity, predicted_activity, valid_mask):
    squared_error = (true_activity - predicted_activity) ** 2
    expanded_mask = valid_mask[..., None]
    denominator = jnp.sum(valid_mask) * true_activity.shape[-1]
    return jnp.sum(squared_error * expanded_mask) / denominator


def geodesic_batch_loss(params, activity_batch, dt_batch, mask_batch):
    predictions = jax.vmap(
        lambda activity, steps: geodesic_model_forward(
            params, activity, steps, CONTEXT_STEPS
        )
    )(activity_batch, dt_batch)
    return masked_mse(activity_batch, predictions, mask_batch)


def free_batch_loss(params, activity_batch, dt_batch, mask_batch):
    predictions = jax.vmap(
        lambda activity, steps: free_model_forward(
            params, activity, steps, CONTEXT_STEPS
        )
    )(activity_batch, dt_batch)
    return masked_mse(activity_batch, predictions, mask_batch)


def parameter_count(params):
    return int(sum(np.size(value) for value in jax.tree_util.tree_leaves(params)))

In [ ]:
# ==============================================================================
# 3. Train both models on identical E65 trial batches
# ==============================================================================

activity = jnp.asarray(e65["activity"])
time_steps = jnp.asarray(e65["time_steps"])
valid_mask = jnp.asarray(e65["valid_mask"])
train_indices = e65["train_indices"]
heldout_indices = e65["heldout_indices"]
n_neurons = e65["activity"].shape[-1]

if BATCH_SIZE > len(train_indices):
    raise ValueError("BATCH_SIZE cannot exceed the number of training trials.")


def checkpoint_path(model_name):
    return OUTPUT_DIR / f"checkpoint_{model_name}.pkl"


def save_checkpoint(
    model_name, params, optimizer_state, next_epoch, history, rng
):
    path = checkpoint_path(model_name)
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    payload = {
        "model_name": model_name,
        "latent_dim": LATENT_DIM,
        "context_steps": CONTEXT_STEPS,
        "encoder_type": ENCODER_TYPE,
        "gru_hidden_dim": GRU_HIDDEN_DIM,
        "n_neurons": int(n_neurons),
        "seed": SEED,
        "heldout_fraction": HELDOUT_FRACTION,
        "max_trials": MAX_TRIALS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LR,
        "x64": ENABLE_X64,
        "next_epoch": int(next_epoch),
        "params": jax.device_get(params),
        "optimizer_state": jax.device_get(optimizer_state),
        "history": np.asarray(history, dtype=np.float64),
        "rng_state": rng.bit_generator.state,
    }
    with temporary_path.open("wb") as handle:
        pickle.dump(payload, handle, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(temporary_path, path)
    print(f"Saved {model_name} checkpoint at epoch {next_epoch}: {path}")


def restore_checkpoint(model_name, params, optimizer_state, rng):
    path = checkpoint_path(model_name)
    if not RESUME or not path.exists():
        return params, optimizer_state, 0, [], rng

    with path.open("rb") as handle:
        payload = pickle.load(handle)
    expected = {
        "model_name": model_name,
        "latent_dim": LATENT_DIM,
        "context_steps": CONTEXT_STEPS,
        "encoder_type": ENCODER_TYPE,
        "gru_hidden_dim": GRU_HIDDEN_DIM,
        "n_neurons": int(n_neurons),
        "seed": SEED,
        "heldout_fraction": HELDOUT_FRACTION,
        "max_trials": MAX_TRIALS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LR,
        "x64": ENABLE_X64,
    }
    legacy_defaults = {
        "encoder_type": "flattened_mlp",
        "gru_hidden_dim": 128,
    }
    for key, value in expected.items():
        actual = payload.get(key, legacy_defaults.get(key))
        if actual != value:
            raise ValueError(
                f"Checkpoint {path} has {key}={actual!r}; "
                f"this run requires {value!r}. Use a different output "
                "directory or disable resume."
            )

    params = jax.tree_util.tree_map(jnp.asarray, payload["params"])
    optimizer_state = jax.tree_util.tree_map(
        jnp.asarray, payload["optimizer_state"]
    )
    rng.bit_generator.state = payload["rng_state"]
    start_epoch = int(payload["next_epoch"])
    history = np.asarray(payload["history"], dtype=np.float64).tolist()
    print(f"Resuming {model_name} from epoch {start_epoch}: {path}")
    return params, optimizer_state, start_epoch, history, rng


def train_model(model_name, initial_params, loss_function):
    optimizer = optax.chain(
        optax.clip_by_global_norm(1.0),
        optax.adam(LR),
    )
    optimizer_state = optimizer.init(initial_params)

    @jax.jit
    def train_step(params, current_optimizer_state, y_batch, dt_batch, mask_batch):
        loss, gradients = jax.value_and_grad(loss_function)(
            params, y_batch, dt_batch, mask_batch
        )
        updates, next_optimizer_state = optimizer.update(
            gradients, current_optimizer_state, params
        )
        next_params = optax.apply_updates(params, updates)
        return next_params, next_optimizer_state, loss

    rng = np.random.default_rng(SEED)
    params, optimizer_state, start_epoch, history, rng = restore_checkpoint(
        model_name, initial_params, optimizer_state, rng
    )
    print(f"\nTraining {model_name} model...")
    started = time.time()

    for epoch in range(start_epoch, EPOCHS):
        # Preserve the original notebook behavior: one random trial batch/epoch.
        selected = rng.choice(train_indices, BATCH_SIZE, replace=False)
        next_params, next_optimizer_state, loss = train_step(
            params,
            optimizer_state,
            activity[selected],
            time_steps[selected],
            valid_mask[selected],
        )
        loss_value = float(loss)

        if not np.isfinite(loss_value):
            print(f"Non-finite loss at epoch {epoch}; stopping {model_name}.")
            break
        params = next_params
        optimizer_state = next_optimizer_state
        history.append(loss_value)
        if epoch % LOG_EVERY == 0 or epoch == EPOCHS - 1:
            print(
                f"{model_name} epoch {epoch:04d} | "
                f"masked reconstruction MSE: {loss_value:.8f}"
            )
        if (
            (epoch + 1) % CHECKPOINT_EVERY == 0
            or epoch == EPOCHS - 1
        ):
            save_checkpoint(
                model_name,
                params,
                optimizer_state,
                epoch + 1,
                history,
                rng,
            )

    print(f"{model_name} optimization: {time.time() - started:.2f} seconds.")
    return params, np.asarray(history, dtype=np.float64)


initial_key = jax.random.PRNGKey(SEED)
initial_geodesic_params = init_geodesic_params(
    initial_key, n_neurons, CONTEXT_STEPS
)
initial_free_params = init_free_params(
    initial_key, n_neurons, CONTEXT_STEPS
)

print(f"Geodesic parameters: {parameter_count(initial_geodesic_params)}")
print(f"Free parameters:     {parameter_count(initial_free_params)}")

geodesic_params, geodesic_history = train_model(
    "geodesic",
    initial_geodesic_params,
    geodesic_batch_loss,
)
free_params, free_history = train_model(
    "free",
    initial_free_params,
    free_batch_loss,
)

In [ ]:
# ==============================================================================
# 4. Leakage-free heldout forecast evaluation
# ==============================================================================

predict_geodesic = jax.jit(
    jax.vmap(
        lambda neural_sequence, steps: geodesic_model_forward(
            geodesic_params, neural_sequence, steps, CONTEXT_STEPS
        )
    )
)
predict_free = jax.jit(
    jax.vmap(
        lambda neural_sequence, steps: free_model_forward(
            free_params, neural_sequence, steps, CONTEXT_STEPS
        )
    )
)

geodesic_prediction = np.asarray(predict_geodesic(activity, time_steps))
free_prediction = np.asarray(predict_free(activity, time_steps))


def reconstruction_metrics(indices, prediction, start_step=0, stop_step=None):
    selected_true = []
    selected_prediction = []
    for trial_index in np.asarray(indices):
        length = int(e65["lengths"][trial_index])
        stop = length if stop_step is None else min(int(stop_step), length)
        if stop <= start_step:
            continue
        selected_true.append(
            e65["activity"][trial_index, start_step:stop].reshape(-1)
        )
        selected_prediction.append(
            prediction[trial_index, start_step:stop].reshape(-1)
        )

    true_values = np.concatenate(selected_true)
    predicted_values = np.concatenate(selected_prediction)
    residual = true_values - predicted_values
    denominator = np.sum((true_values - true_values.mean()) ** 2)
    correlation = np.corrcoef(true_values, predicted_values)[0, 1]
    return {
        "mse": float(np.mean(residual ** 2)),
        "r2": float(1.0 - np.sum(residual ** 2) / denominator),
        "r": float(correlation),
        "n": int(true_values.size),
    }


results = {
    "geodesic": {
        "train": reconstruction_metrics(
            train_indices, geodesic_prediction
        ),
        "heldout_context": reconstruction_metrics(
            heldout_indices,
            geodesic_prediction,
            start_step=0,
            stop_step=CONTEXT_STEPS,
        ),
        "forecast": reconstruction_metrics(
            heldout_indices,
            geodesic_prediction,
            start_step=CONTEXT_STEPS,
        ),
    },
    "free": {
        "train": reconstruction_metrics(
            train_indices, free_prediction
        ),
        "heldout_context": reconstruction_metrics(
            heldout_indices,
            free_prediction,
            start_step=0,
            stop_step=CONTEXT_STEPS,
        ),
        "forecast": reconstruction_metrics(
            heldout_indices,
            free_prediction,
            start_step=CONTEXT_STEPS,
        ),
    },
}

print("\n" + "=" * 72)
print("E65 GEODESIC VERSUS FREE-DYNAMICS COMPARISON")
print("=" * 72)
print(f"{'Metric':<22} | {'Geodesic':>20} | {'Free':>20}")
print("-" * 72)
for label, section, metric in [
    ("Train MSE", "train", "mse"),
    ("Train R2", "train", "r2"),
    ("Train r", "train", "r"),
    ("Forecast MSE", "forecast", "mse"),
    ("Forecast R2", "forecast", "r2"),
    ("Forecast r", "forecast", "r"),
]:
    print(
        f"{label:<22} | "
        f"{results['geodesic'][section][metric]:>20.8f} | "
        f"{results['free'][section][metric]:>20.8f}"
    )
print("=" * 72)

# Fixed visualization: the first heldout trial, same neurons/time/color scale.
plot_index = int(heldout_indices[0])
length = int(e65["lengths"][plot_index])
n_plot_neurons = min(50, n_neurons)
true_plot = e65["activity"][plot_index, :length, :n_plot_neurons].T
geodesic_plot = geodesic_prediction[
    plot_index, :length, :n_plot_neurons
].T
free_plot = free_prediction[plot_index, :length, :n_plot_neurons].T
maximum = max(
    float(true_plot.max()),
    float(geodesic_plot.max()),
    float(free_plot.max()),
)
trial_times = e65["times"][plot_index, :length]
context_time = trial_times[CONTEXT_STEPS - 1]

figure, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)
for axis, values, title in zip(
    axes,
    [true_plot, geodesic_plot, free_plot],
    ["True E65", "Geodesic prediction", "Free prediction"],
):
    image = axis.imshow(
        values,
        aspect="auto",
        origin="lower",
        cmap="magma",
        vmin=0.0,
        vmax=maximum,
        extent=[trial_times[0], trial_times[-1], 0, n_plot_neurons],
    )
    axis.axvline(context_time, color="white", linewidth=1.5)
    axis.set_title(title)
    axis.set_xlabel("Time (s)")
axes[0].set_ylabel("Neuron")
figure.colorbar(image, ax=axes, label="Activity", shrink=0.8)
figure.suptitle(
    f"Heldout trial {e65['trial_ids'][plot_index]:g}; "
    f"white line = end of {CONTEXT_STEPS}-frame context"
)
figure.savefig(
    OUTPUT_DIR / "heldout_geodesic_vs_free.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()

summary = {
    "config": {
        "data_path": str(DATA_PATH),
        "latent_dim": LATENT_DIM,
        "context_steps": CONTEXT_STEPS,
        "encoder_type": ENCODER_TYPE,
        "gru_hidden_dim": GRU_HIDDEN_DIM,
        "heldout_fraction": HELDOUT_FRACTION,
        "max_trials": MAX_TRIALS,
        "num_trials": int(len(e65["trial_ids"])),
        "num_train_trials": int(len(train_indices)),
        "num_heldout_trials": int(len(heldout_indices)),
        "n_neurons_raw": int(e65["n_neurons_raw"]),
        "n_neurons_modeled": int(n_neurons),
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LR,
        "dtype": "float64" if ENABLE_X64 else "float32",
        "checkpoint_every": CHECKPOINT_EVERY,
        "resume": RESUME,
        "solver": "rk4",
        "loss": "masked_mse",
        "uses_raw_frame_times": True,
        "uses_position_or_behavior": False,
        "geodesic_parameters": parameter_count(geodesic_params),
        "free_parameters": parameter_count(free_params),
    },
    "geodesic": results["geodesic"],
    "free": results["free"],
}
(OUTPUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2) + "\n")
np.savez_compressed(
    OUTPUT_DIR / "fit_outputs.npz",
    trial_ids=e65["trial_ids"],
    train_indices=train_indices,
    heldout_indices=heldout_indices,
    neuron_mask=e65["neuron_mask"],
    lengths=e65["lengths"],
    times=e65["times"],
    valid_mask=e65["valid_mask"],
    true_activity=e65["activity"],
    geodesic_prediction=geodesic_prediction,
    free_prediction=free_prediction,
    geodesic_mse_history=geodesic_history,
    free_mse_history=free_history,
)
print(f"Saved outputs to {OUTPUT_DIR}")